In [1]:
import pandas as pd
import numpy as np

In [2]:
# Read movie dataset

i_cols= ['movie id', 'movie title' ,'release date','video release date', 'IMDb URL', 'unknown', 'Action', 'Adventure',
    'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
    'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

items = pd.read_csv( 'ml-100k/u.item',sep='|',names=i_cols, encoding='latin-1' )

In [3]:
# Read rating dataset
r_cols=['user_id','movie_id','rating','unix_timestamp']
ratings=pd.read_csv("ml-100k/u.data",sep='\t', names=r_cols)  ## can add encoding='latin-1'

In [68]:
def popularbasedRS(items,ratings,u_genre,movies_count):
    # Create movie Vector

    movie_vec=items[['Action', 'Adventure',
    'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
    'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']]

    # Rename the index from RangeIndex(start=0, stop=1682, step=1) to RangeIndex(start=1, stop=1683, step=1)
    movie_vec.index=range(1,1683)
    movie_vector=movie_vec.T

    # Calculate all means at once
    mean_ratings = ratings.groupby('movie_id')['rating'].mean()
    
    # Map the means directly to movie vector dataset
    movie_vector = movie_vector.copy()
    for i in sorted(ratings.movie_id.unique()):
        if (i in movie_vector.columns): 
            movie_vector[i] = movie_vector[i].astype(float)
            movie_vector.loc[movie_vector[i] > 0,i]= mean_ratings[i]

    ## To get the movie id list for each genre
    x= movie_vector.T
    popular_movies_list=[]
    for genre in x.columns:
        list1= list(x[genre].sort_values(ascending=False).head(movies_count).index)
        popular_movies_list.append(list1)
        #print(f"Recommended movies in {genre} genre are {list1}")
        
    popular_movies_df=pd.DataFrame(popular_movies_list)
    popular_movies = popular_movies_df.T
    popular_movies.columns=x.columns

    ## To get the movie name for each movie id and 
    rows_list=[]
    for genre in x.columns:
        for movieid in popular_movies[genre]:
            movie_name=items[items['movie id'] == movieid]['movie title'].values[0]

            row_entry = {
                'Genre': genre,
                'Movie_ID': movieid,
                'Movie_Title': movie_name
            }
            rows_list.append(row_entry)

    # Convert the list of rows into a DataFrame
    recommendations_df = pd.DataFrame(rows_list)
    
            
    print(f"\nRecommended movies for each genre are\n {recommendations_df}")

    recommendations_df.to_csv('popularbased_recommendations.csv', index=False)
    
    print(f"\nRecommended movies for {u_genre} genre are\n {recommendations_df[recommendations_df['Genre']== u_genre]}")
    
    #movie_name_list=[]

In [70]:
popularbasedRS(items,ratings,'Action',10)  ## Enter the genre and number of top movies to be listed


Recommended movies for each genre are
        Genre  Movie_ID                             Movie_Title
0     Action        50                        Star Wars (1977)
1     Action       127                   Godfather, The (1972)
2     Action       174          Raiders of the Lost Ark (1981)
3     Action       313                          Titanic (1997)
4     Action       172         Empire Strikes Back, The (1980)
..       ...       ...                                     ...
175  Western       203                       Unforgiven (1992)
176  Western       177  Good, The Bad and The Ugly, The (1966)
177  Western       922                         Dead Man (1995)
178  Western        97               Dances with Wolves (1990)
179  Western       470                        Tombstone (1993)

[180 rows x 3 columns]

Recommended movies for Action genre are
     Genre  Movie_ID                      Movie_Title
0  Action        50                 Star Wars (1977)
1  Action       127            G